# Sprint 1 Live — Core-concept workshops
Use this instructor Colab for LS01–LS04, in a separate practice workspace. Campus Sprints 1–3 use VS Code and two terminals; Sprint 1 uses its Campus Colab only for checkpoint verification. No Campus ZIP or business assignment is needed here.

The public diagnosis example is a small vehicle for the concept. Inspect the same `service/fieldcare/` source layout used locally. The supplied `live_*.py` modules make each registration/attachment visible and run in separate owned processes; in Campus those lines go in `main.py`. Each start stops only our previous process. Never restart during a quota-renewal experiment.

Run setup once. It clones `main`, installs the lock, preserves an existing marked Live folder, and performs no provider call. `ServiceProcess` starts a loopback Uvicorn child with a bounded readiness check. A Colab `127.0.0.1` address belongs to Colab, not your laptop. No foreground Uvicorn call or public tunnel is needed.

See [local setup](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/local-development.md) and [Live handoff](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/live-handoff.md).

**Continue in a fresh runtime:** download the Live ZIP and save the notebook after each session. Upload your own ZIP through the Colab Files panel (or use its local path in Jupyter), set `LIVE_ARCHIVE` below, and fill both `RESTORED_MODELS` from your saved model-choice record. Choose a `LIVE_FOLDER` that does not exist. Setup validates the ZIP and restores your edited source, router registrations and selected evidence without running it. Review that source before starting a service. After a successful restore, clear `LIVE_ARCHIVE` before rerunning setup in the same runtime so it reuses the marked folder. Credentials and indexes are excluded: re-enter provider credentials and prepare the index only when the session requires them. A Campus checkpoint cannot be restored here.


In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, secrets, time
import httpx
REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))
from module_b.live import prepare_live_workspace, export_live_workspace, restore_live_workspace
from module_b.runtime import ServiceProcess
# On a fresh runtime, upload your own Live ZIP using the Files panel.
LIVE_ARCHIVE = ""  # Example: "/content/sprint-1-live-123.zip"; keep empty for new practice.
LIVE_FOLDER = Path.cwd() / "fieldcare-sprint-1-live"
# When restoring, explicitly select the same approved models used in that work.
RESTORED_MODELS = {"OPENROUTER_MODEL": "", "OPENROUTER_MODEL_V2": ""}
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
if LIVE_ARCHIVE:
    if any(model not in CHAT_MODELS for model in RESTORED_MODELS.values()):
        raise ValueError("Fill both RESTORED_MODELS with your saved approved model choices before restoring.")
    PROJECT = restore_live_workspace(LIVE_ARCHIVE, LIVE_FOLDER)
else:
    PROJECT = prepare_live_workspace(REPO, LIVE_FOLDER)
CALLERS = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
caller_keys = {name: secrets.token_urlsafe(32) for name in CALLERS}
previous_models = {name: globals().get("service_env", {}).get(name) for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
service_env = {variable: caller_keys[name] for name, variable in CALLERS.items()}
service_env["FIELDCARE_WORK_DIR"] = str((PROJECT / "var").resolve())
for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2"):
    service_env[name] = (RESTORED_MODELS[name] if LIVE_ARCHIVE else
                         previous_models[name] or os.environ.get(name, DEFAULT_CHAT_MODEL))
    if service_env[name] not in CHAT_MODELS:
        raise ValueError(f"Select an approved {name}.")
print("Configured models:", {name: service_env[name] for name in RESTORED_MODELS})
if globals().get("service") is not None:
    service.stop()
service = None
supported = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
incomplete = {"question": "Which filter checks are documented?"}
print("Live workspace:", PROJECT)
print("Source main:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
print("No provider or index is needed for health, validation and clarification.")

from module_b.live import start_live_service

## LS01 — Predict, call, locate
Explain method/path, JSON body, status and response fields before executing. First inspect the server entry, route and schemas; predict clarification versus validation. No provider key is needed.

In [ ]:
print((PROJECT / "service/fieldcare/main.py").read_text())
print((PROJECT / "service/fieldcare/routes.py").read_text())
print((PROJECT / "service/fieldcare/schemas.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "main", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    print("health", client.get("/health").json())
    for label, body in [("missing optional context", incomplete), ("invalid input", {})]:
        response = client.post("/v1/diagnose", json=body)
        print(label, response.status_code, response.json())

Trace `clarification_for` before `graph_for`, then follow retrieval and `client.chat` in the source. Health is process/preparation state. Optional provider work follows only after you can explain 200/clarification and 422.

### Optional provider preparation — after the deterministic checks
The same explicit preparation as Campus: stop the service, load the private provider key, then run `python -m fieldcare.prepare_index`. Embedding the supplied synthetic documents uses provider quota. Set the switch only when you intend to do this. A healthy process alone proves neither provider access nor answer quality. In local Campus, the key belongs in private `.env`; in this instructor Colab, use Colab Secrets or the hidden prompt. Restart the relevant service cell after preparation.

In [ ]:
PREPARE_PROVIDER = False
if PREPARE_PROVIDER:
    if service is not None:
        service.stop()
    from module_b.openrouter import require_openrouter_key
    require_openrouter_key(prompt=True)
    preparation_env = {**os.environ, **service_env,
                       "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
    subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"],
                   cwd=PROJECT, env=preparation_env, check=True)
else:
    print("Provider preparation pending; deterministic checks remain available.")

In [ ]:
RUN_PROVIDER = False
if RUN_PROVIDER:
    service = start_live_service(PROJECT, "main", service_env, service)
    with httpx.Client(base_url=service.base_url, timeout=120) as client:
        response = client.post("/v1/diagnose", json=supported)
        print(response.status_code, response.json())
else:
    print("Generated-answer evidence pending.")

## LS02 — Register, bind, check
Read the supplied `brief_routes.py` and the registration in `live_extension.py`. The brief route is the instructor demonstration. Locate its schemas, early clarification, route-owned prompt and `graph_for(system_prompt=...)`. Keep it and the baseline route.

**Build your own operation.** A second technician screen needs a checklist of documented checks followed by one sentence about what still needs on-site verification. In the Files editor create `checklist_schemas.py` with explicitly named request/response classes inheriting `DiagnosticRequest`/`DiagnosticResponse`. Keep the existing `1–2000` question limit, optional equipment, four output fields and clarification states. Create `checklist_routes.py` using the brief route as a pattern; bind those classes and your own `CHECKLIST_PROMPT` to `POST /v1/diagnose-checklist`. Preserve the shared safety instructions, deterministic checks and provider-failure handling. Import and include your router once in `live_extension.py`.

Your new path, contract declarations, prompt binding and registration must be learner-authored. Merely inspecting the brief route or deciding no change is needed does not finish this task. Use the actual source diff and contract card as evidence. Do not modify the supplied brief operation. Restart `ls02-start` after editing; then set `ROUTE_TO_REVIEW` to your new path and run `code-06`.

In [ ]:
print((PROJECT / "service/fieldcare/brief_routes.py").read_text())
print((PROJECT / "service/fieldcare/live_extension.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_extension", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
ROUTE_TO_REVIEW = "/v1/diagnose-brief"  # Demo first; change to your /v1/diagnose-checklist after authoring.
RUN_PROVIDER = False
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    for path in [ROUTE_TO_REVIEW, "/v1/diagnose"]:
        for label, body in [("clarification", incomplete), ("invalid", {})]:
            response = client.post(path, json=body)
            print(path, label, response.status_code, response.json())
        if RUN_PROVIDER:
            response = client.post(path, json=supported)
            print(path, "provider comparison", response.status_code, response.json())

Compare each response with the contract card. Generated wording varies, so a single response cannot establish that a prompt improved answer quality. Record one expectation the caller can rely on from the contract and one claim that requires evidence beyond schema validation.

## LS03 — Two contracts and independent model selection
Keep your LS02 modules and registration: `live_versions.py` imports `live_extension.py`, so the checklist remains callable. Read the pilot schema, prompt and model argument before starting `/v2/diagnose`. V1 accepts 2000 characters and uses `OPENROUTER_MODEL`; the pilot accepts 600 and passes `openrouter_model_v2()` into `graph_for`. Each setting is independently validated against the course-approved list; an unset setting has its own default.

First run `ls03-model-before` to capture the selected models and source hashes. In `ls03-model-change`, choose another approved model for v2 only. That cell verifies that v1 and the source bindings did not change. Restart `ls03-start`, then repeat the contract checks. Model selection and source checks do not claim a real provider ran.

In [ ]:
print((PROJECT / "service/fieldcare/pilot_schemas.py").read_text())
print((PROJECT / "service/fieldcare/pilot_routes.py").read_text())
print((PROJECT / "service/fieldcare/live_versions.py").read_text())

In [ ]:
import hashlib
# These are non-secret configuration values and source digests, not provider observations.
binding_paths = sorted((PROJECT / "service/fieldcare").glob("*.py"))
source_before_model_change = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in binding_paths}
models_before = {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
print("Before:", models_before)
print("Approved choices:", sorted(CHAT_MODELS))


In [ ]:
PILOT_MODEL = ""  # Choose a different approved model from the printed list.
if PILOT_MODEL not in CHAT_MODELS or PILOT_MODEL == models_before["OPENROUTER_MODEL_V2"]:
    raise ValueError("Choose another course-approved pilot model before continuing.")
service_env["OPENROUTER_MODEL_V2"] = PILOT_MODEL
assert service_env["OPENROUTER_MODEL"] == models_before["OPENROUTER_MODEL"]
assert source_before_model_change == {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in binding_paths}
print("After:", {name: service_env[name] for name in models_before})
selection_env = {**os.environ, **service_env,
                 "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
subprocess.run([sys.executable, "-m", "clients.version_models"], cwd=PROJECT, env=selection_env, check=True)
print("V1 selection, prompts and schemas stayed unchanged. Restart both routes for the contract checks.")


In [ ]:
service = start_live_service(PROJECT, "live_versions", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
OLD_PATH, NEW_PATH = "/v1/diagnose", "/v2/diagnose"
long_request = {"question": "filter " + "x" * 693}
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    for path in [OLD_PATH, NEW_PATH]:
        for label, body in [("short", incomplete), ("700 characters", long_request), ("invalid", {})]:
            response = client.post(path, json=body)
            print(path, label, response.status_code, response.json().get("status"))
# Verify the LS02 registration survived the later composition.
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    response = client.post("/v1/diagnose-checklist", json=incomplete)
    print("Your retained LS02 operation:", response.status_code, response.json())
    if response.status_code == 404:
        print("LS02 work is incomplete: create and register your own operation before review.")


For independent review, make the v2-only model change first and retain the before/after comparison. Then change only the pilot formatting instruction in `pilot_routes.py` as a separate edit, restart `ls03-start` and repeat `code-09`. Keep schemas, v1 prompt/model and the LS02 operation unchanged. Record each change separately so its effect can be attributed. If provider setup is ready, compare real supported responses separately. Retain v1 until its callers can migrate; deprecation is a notice, not removal.

## LS04 — Demonstrate and review

Demonstrate the operation you authored in LS02, with its named contract and versioned registration. Show an accepted request and an invalid request, then the original-route regression. Ask a reviewer where each request is rejected or handled. Also retain the LS03 v1/v2 contract and model-binding comparison; the supplied pilot alone does not demonstrate that you added an endpoint.

In [ ]:
REVIEW_PATH = "/v1/diagnose-checklist"  # Your LS02 extension, retained in live_versions.
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    for path, body in [(REVIEW_PATH, incomplete), (REVIEW_PATH, {}), ("/v1/diagnose", long_request)]:
        response = client.post(path, json=body)
        print(path, response.status_code, response.json())

In your saved notebook notes, record the route and contract you showed, the evidence you observed, one limitation of that evidence, and a repair you made or why no change was needed. Save the notebook and keep the Live ZIP separate from the Campus checkpoint.

## Save your Live practice
Save your individual explanation in the notebook and selected safe observations in `evidence/`. The export includes source and selected evidence; it excludes `.env`, the index and raw logs. This is public Live practice. Your existing VS Code Campus project remains your authoring copy; repeat a relevant check there and use its own export for assessment.

In [ ]:
if service is not None:
    service.stop()
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/live-models.json").write_text(json.dumps(
    {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}, indent=2) + "\n")
print("Save these configured model choices with your notebook:",
      {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")})
CHECKPOINT_OUT = PROJECT.parent / f"sprint-1-live-{time.time_ns()}.zip"
print("Download:", export_live_workspace(PROJECT, CHECKPOINT_OUT))